# Rich Neale's Blocking Diagnostic (notebook version)

### How to run
The run settings are in two text cells in Section 1, written in the same formats as the MDTF files:
the runtime config (`input_*.yml`) and the POD settings (`settings.jsonc`). The notebook detects how it is being run,
and in every mode writes the same output as an MDTF framework run: `<WORK_DIR>/MDTF_output/` with `index.html`,
the POD page `blocking_neale_nb/blocking_neale_nb.html`, figures and logs, plus `<WORK_DIR>/MDTF_output.tar`
when `make_variab_tar` is True.

- **Interactive (Jupyter)**: edit those two cells and run all cells. The export cell writes them out as
  `input_<case>.yml` and `settings.jsonc` and prints the commands to repeat the run in batch.
  Save the notebook (Ctrl-S) before the last cell runs (or save and re-run it): that cell writes the
  executed notebook `blocking_main_ipynb.html` from the saved file, before making the tar.
- **Headless (no Jupyter)**, from the MDTF-diagnostics directory:
  `python -m src.standalone_html run diagnostics/blocking_neale_nb/blocking_main.ipynb --config <input.yml> [--settings <settings.jsonc>]`
  (without `--settings`, `settings.jsonc` in this directory is used).
- **MDTF framework**: `mdtf -f <input.yml>` with `blocking_neale_nb` in `pod_list`. The framework reads
  `settings.jsonc` from this directory, preprocesses the model data and builds the output itself.

Runs in a standard MDTF python environment (`_MDTF_python3_base`); model data are read from the
ESM-intake catalog with `intake-esm`.

### Section 1: Prepare POD Settings
Set the parameters of the POD and the runtime configuration
### Section 2: Load model data (catalog, or the framework's preprocessed files)
### Section 3: Diagnostics Computation
### Section 4: Results Visualization
### Section 5: Output (POD page, index page, logs, tar)
More about this POD: `doc/blocking_neale_nb.rst`

More details on the development process:
[MDTF Planning Document](https://docs.google.com/document/d/1P8HqL8O5304qwR3ik9RmgFDwSWwlkPgOjnp39PIkLfY/edit?usp=sharing)

In [2]:
# Import General Package

import importlib

import sys
import yaml
import json

import os


## Section 1: Settings
Paths and imports, then the POD settings and run settings as text, then the settings resolved for this run.

In [ ]:
#
# Framework: Setup paths to import MDTF code as modules
#
POD_NAME = "blocking_neale_nb"
NOTEBOOK = "blocking_main.ipynb"
import time
RUN_START = time.time()   # the last cell checks the notebook was saved after this

# POD code directory: $POD_HOME when run by the MDTF framework (which runs a copy of this
# notebook in the output directory), otherwise the directory this notebook is in.
pod_code_dir = os.path.abspath(os.environ.get("POD_HOME", "."))
module_path = os.path.abspath(os.path.join(pod_code_dir, "../../"))  # MDTF-diagnostics/
print(f"Using MDTF framework version {module_path}")
for p in (module_path, pod_code_dir):
    if p not in sys.path:
        sys.path.append(p)
from src import standalone_html   # run settings, export, and stand-alone POD web page

# The MDTF framework sets MPLBACKEND=Agg for every POD. In a Jupyter kernel, with the
# matplotlib 3.8 / IPython 9 in the MDTF python envs, that makes importing pyplot fail
# (backend2gui no longer in IPython), so use the notebook's own (inline) backend.
if os.environ.get("MPLBACKEND", "").lower() == "agg":
    os.environ.pop("MPLBACKEND")

import blocking_utils as block_utils
import blocking_figs as block_figs

In [ ]:
#
# POD settings, in settings.jsonc format (JSON with // comments).
# Used when running interactively. Headless and framework runs read settings.jsonc
# from the POD directory instead; the export cell below writes this text to a file.
#
pod_settings_jsonc = r"""
// MDTF POD settings for blocking_neale_nb (Rich Neale's blocking diagnostic, notebook version)
//
// JSON with "//" comments (JSONC). The same text is held in the notebook's
// POD settings cell; the notebook's export cell writes it back out as this file.
{
  "settings" : {
    // Notebook run by the framework (jupyter nbconvert --execute)
    "driver" : "blocking_main.ipynb",
    "long_name" : "Rich Neale's blocking diagnostic",
    "convention": "cesm",
    "description" : "Rich Neale's blocking diagnostic (notebook version)",

    // POD options. The framework passes these to the notebook as environment
    // variables, so every value is a string; lists are comma-separated.
    "pod_env_vars" : {
      // Season for the blocking frequency: DJF, MAM, JJA, SON or ANN
      "MDTF_BLOCKING_SEASON"      : "DJF",
      // Hemisphere: only nhem is implemented
      "MDTF_BLOCKING_HEMISPHERE"  : "nhem",
      // Obs and model ensembles to compare with the MDTF case(s): ERA5, MERRA (obs);
      // timeslice3 (MDTF v3 timeslice CESM run, 1995-2014) and CESM2 (large ensemble)
      // (model ensembles). Their blocking frequencies are shipped as digested data in
      // the POD's obs data ($OBS_DATA) for the members and years below, so their raw
      // data are not read. Other choices need raw daily Z500 through
      // MDTF_BLOCKING_RAW_CATALOG; also known, but untested with this version: ERAI,
      // CESM1, E3SMv1, E3SMv2, EAMv2, CAM6 (see RAW_DATA in blocking_utils.py).
      "MDTF_BLOCKING_ENSEMBLES"   : "ERA5,MERRA,timeslice3,CESM2",
      // Members to use from each ensemble above (same order)
      "MDTF_BLOCKING_ENS_MEMBERS" : "1,1,1,5",
      // Years for the obs and model ensembles: one year for all, or one per entry in
      // MDTF_BLOCKING_ENSEMBLES (same order). Blank: use the MDTF case years.
      // (The NCL version of this POD compared against 1979-1999.)
      "MDTF_BLOCKING_REF_FIRSTYR" : "1979,1979,1995,1979",
      "MDTF_BLOCKING_REF_LASTYR"  : "1999,1999,2014,1999",
      // Blocking frequencies are saved as netCDF and reused when a later run needs the
      // same index, dataset, members, years and season. Results from an older version
      // of the calculation are recomputed automatically.
      // Recompute anyway: "False", "True" (everything) or dataset names, e.g. "CESM2,ERA5"
      "MDTF_BLOCKING_RECOMPUTE"   : "False",
      // Where to keep them. Blank: this run's output (model/netCDF). Set a shared
      // directory to reuse results across runs and cases. The shipped results in
      // $OBS_DATA are read after this directory and are never written to.
      "MDTF_BLOCKING_CACHE_DIR"   : "",
      // ESM-intake catalog of the raw daily Z500 of the comparison datasets, used only
      // for results not in the cache or $OBS_DATA (made by blocking_utils.make_raw_catalog;
      // NCAR GLADE only). Blank: none, so only the shipped choices work.
      "MDTF_BLOCKING_RAW_CATALOG" : ""
    },

    // The framework picks the conda env from the first language with a
    // non-empty package list, so the list must not be empty.
    "runtime_requirements": {
      "python3": ["numpy", "pandas", "xarray", "scipy", "matplotlib", "cartopy", "intake-esm", "nbconvert"]
    }
  },

  "dimensions": {
    "lat": {
      "standard_name": "latitude",
      "units": "degrees_north",
      "axis": "Y"
    },
    "lon": {
      "standard_name": "longitude",
      "units": "degrees_east",
      "axis": "X"
    },
    "lev": {
      "standard_name": "air_pressure",
      "units": "hPa",
      "positive": "down",
      "axis": "Z"
    },
    "time": {"standard_name": "time"}
  },

  "varlist" : {
    "zg": {
      "path_variable" : "MODEL_DATA_PATH",
      "standard_name" : "geopotential_height",
      "units": "m",
      "realm" : "atmos",
      "frequency" : "day",
      "dimensions": ["time", "lat", "lon"],
      "scalar_coordinates": {"lev" : 500}
    }
  }
}
"""


In [ ]:
#
# Run settings, in MDTF runtime config (input_*.yml) format.
# Used when running interactively. A headless run reads the file named by
# $MDTF_RUNTIME_CONFIG instead; the export cell below writes this text to a file.
#
runtime_yml = r"""
# List of POD(s) to run (used by the MDTF framework)
pod_list:
   - "blocking_neale_nb"

# Case list entries (must be unique IDs for each simulation)
# The MDTF framework only finds catalog files whose path contains the case name
case_list:
  "mdtf_timeslice_v4.ne30L58" :
    model: "CESM"
    convention: "CESM"
    startdate: "00010101000000"
    enddate:   "00201231000000"

### Data location settings ###
# Required: full or relative path to ESM-intake catalog header file
DATA_CATALOG: "/glade/campaign/cgd/amp/bundy/mdtf/cesm_mdtfv4_timeslice_public/mdtf_timeslice_v4.ne30L58/esm_catalog_CESM_timeslice_mdtfv4_ne30.v43.20261102.json"
# Optional: Parent directory containing observational data used by individual PODs.
OBS_DATA_ROOT: "/glade/work/bundy/mdtf/inputdata/obs_data"
# Required: Working directory location. The POD output goes in WORK_DIR/blocking_neale_nb
WORK_DIR: "/glade/work/bundy/mdtf/outdir/blocking_nb"
# Optional: Location to write final output if you don't want it in the WORK_DIR
OUTPUT_DIR: ""

### Environment Settings (used by the MDTF framework) ###
conda_root: "/glade/work/bundy/miniconda2/"
conda_env_root: "/glade/work/bundy/miniconda2/envs.MDTFv4.3.20260924/"
micromamba_exe: ""

### Data type settings ###
large_file: False

### Output Settings ###
# Set to true to have PODs save postscript figures in addition to bitmaps.
save_ps: False
save_pp_data: True
translate_data: True
make_variab_tar: True    # also write <WORK_DIR>/MDTF_output.tar, as the framework does
overwrite: True
make_multicase_figure_html: False
### Developer settings ###
run_pp: True
user_pp_scripts:
  - ""
"""


In [ ]:
#
# Resolve the run settings for this run mode and show them
#   interactive : the two text cells above
#   headless    : $MDTF_RUNTIME_CONFIG file, and $MDTF_POD_SETTINGS or ./settings.jsonc
#   framework   : the MDTF framework's environment variables and case_info.yml
#
run = standalone_html.load_run_settings(POD_NAME, pod_code_dir,
                                        runtime_yml_text=runtime_yml,
                                        pod_settings_text=pod_settings_jsonc)
settings_dict = run.settings
case_info = run.runtime_cfg
pod_env = run.pod_env_vars
print(f"Run mode: {run.mode}\n")
print(yaml.dump(case_info, sort_keys=False))
print(json.dumps(pod_env, indent=2))

# POD options (pod_env_vars are strings; lists are comma-separated)
block_season = pod_env["MDTF_BLOCKING_SEASON"]
block_diag_hem = [s.strip() for s in pod_env["MDTF_BLOCKING_HEMISPHERE"].split(",")]
ens_names = [s.strip() for s in pod_env["MDTF_BLOCKING_ENSEMBLES"].split(",")]
ens_mem_num = [int(s) for s in pod_env["MDTF_BLOCKING_ENS_MEMBERS"].split(",")]
assert len(ens_names) == len(ens_mem_num), "MDTF_BLOCKING_ENSEMBLES and MDTF_BLOCKING_ENS_MEMBERS differ in length"

# Case period (all cases are assumed to cover the same period)
first_case = next(iter(case_info["case_list"].values()))
start_year = standalone_html.year_of(first_case["startdate"])
end_year = standalone_html.year_of(first_case["enddate"])

# Years for the obs and model ensembles: one value for all, or one per ensemble
# (comma-separated, same order as ens_names). Blank: same as the case.
def _per_ensemble(text, default):
    vals = [v.strip() for v in str(text).split(",")] if str(text).strip() else [default]
    if len(vals) == 1:
        vals = vals * len(ens_names)
    assert len(vals) == len(ens_names), f"{text!r}: give one year, or one per ensemble {ens_names}"
    return [v or default for v in vals]
ref_start_years = _per_ensemble(pod_env.get("MDTF_BLOCKING_REF_FIRSTYR", ""), start_year)
ref_end_years = _per_ensemble(pod_env.get("MDTF_BLOCKING_REF_LASTYR", ""), end_year)

# Output follows the MDTF layout: <WORK_DIR>/blocking_neale_nb/model/ (figures) and model/netCDF/
pod_work_dir = run.pod_work_dir
standalone_html.make_pod_work_dirs(pod_work_dir)
fig_dir = os.path.join(pod_work_dir, "model")
fout_dir = os.path.join(pod_work_dir, "model", "netCDF")

# Caching of blocking frequencies (see the POD settings cell)
recompute = block_utils.parse_recompute(pod_env.get("MDTF_BLOCKING_RECOMPUTE", "False"))
cache_dir = os.path.expandvars(os.path.expanduser(pod_env.get("MDTF_BLOCKING_CACHE_DIR", "").strip())) or fout_dir
# Digested obs/ensemble results shipped with the POD (<OBS_DATA_ROOT>/blocking_neale_nb), read-only.
# A dataset found there is not read at all; only the case(s) are read, from the catalog.
ref_dir = run.obs_data
# Raw-data catalog for the comparison datasets: only used for results not found above
raw_catalog = os.path.expandvars(os.path.expanduser(pod_env.get("MDTF_BLOCKING_RAW_CATALOG", "").strip()))
print(f"\nSeason {block_season}, hemisphere {block_diag_hem}, case years {start_year}-{end_year}, "
      "obs/ensemble years " + ", ".join(f"{e} {a}-{b}" for e, a, b in zip(ens_names, ref_start_years, ref_end_years)))
print(f"Ensembles {ens_names}, members {ens_mem_num}")
print(f"Figures -> {fig_dir}\nnetCDF  -> {fout_dir}")
print(f"Cache   -> {cache_dir}  (recompute: {recompute})")
print(f"Digested obs/ensembles <- {ref_dir or '(none: OBS_DATA_ROOT not set)'}")


In [ ]:
#
# Export the run settings so this run can be repeated in batch (interactive runs only).
# Writes input_<case>.yml and settings.jsonc to the POD output directory and prints the commands.
# install_pod_settings = True also copies settings.jsonc into the POD code directory,
# which is where the MDTF framework reads it (overwrites that file; it is in git).
#
export_settings = True
install_pod_settings = False
if export_settings and run.mode == "interactive":
    exported = standalone_html.export_run_settings(
        run, notebook=os.path.join(pod_code_dir, NOTEBOOK),
        install_pod_settings=install_pod_settings)


In [ ]:
# Name of the POD variable in the model data. CESM catalogs list Z500 (zg at 500 hPa);
# other conventions may list zg on pressure levels, in which case 500 hPa is selected.
mapping = {
    "zg": "Z500",
}


## Section 2: Load model data
Stand-alone runs read each case from the ESM-intake catalog (`DATA_CATALOG`), with the same query the MDTF
framework uses (case name in the file path, frequency, realm and standard_name from the varlist).
Framework runs read the framework's preprocessed file listed in its `case_info.yml`.

In [ ]:
import xarray as xr

var_settings = settings_dict['varlist']['zg']
case_data = {}    # case name -> Dataset with Z500
data_files = []   # model files read (for the data log)
for case_name, c in run.case_list.items():
    if run.mode == "framework":
        files = [c['MODEL_DATA_PATH']]
        ds_case = xr.open_dataset(files[0], chunks={'time': 365}).rename({c.get('zg_var', 'zg'): 'Z500'})
    else:
        ds_case, files = standalone_html.open_catalog_var(
            run.catalog, case_name, var_settings, names=[mapping['zg'], 'zg'], out_name='Z500')
    case_data[case_name] = ds_case
    data_files += files
    print(f"Case {case_name}: {files}")
    print(ds_case['Z500'])


## Section 3: Diagnostics computation

### Obs and model ensembles to compare against

In [ ]:
# Table (block_meta) of the obs and model ensembles to compare against: type, members, years,
# run names and files. ens_names, ens_mem_num and the years come from the settings cells.
importlib.reload(block_utils) # pick up edits to blocking_utils.py
ens_ystart = ref_start_years
ens_yend   = ref_end_years
block_meta = block_utils.ens_setup(ens_names,ens_mem_num,ens_ystart,ens_yend,raw_catalog)

In [27]:
#See what is in the block_meta structure

print("block_meta type:", type(block_meta))
print("block_meta columns:", block_meta.columns if hasattr(block_meta, 'columns') else None)
print("block_meta index:", block_meta.index if hasattr(block_meta, 'index') else None)
print("block_meta head:")
print(block_meta.head())
print("block_meta info:")
if hasattr(block_meta, 'info'):
    block_meta.info()

block_meta type: <class 'pandas.core.frame.DataFrame'>
block_meta columns: Index(['Ensemble Type', 'Ensemble Size', 'Start Year', 'End Year', 'Run Name',
       'Run File'],
      dtype='object')
block_meta index: Index(['ERA5', 'MERRA', 'CESM2'], dtype='object')
block_meta head:
      Ensemble Type  Ensemble Size Start Year End Year  \
ERA5            obs              1       1995     1996   
MERRA           obs              1       1995     1996   
CESM2         model              2       1995     1996   

                                                Run Name  \
ERA5                                              [ERA5]   
MERRA                                            [MERRA]   
CESM2  [b.e21.BHISTcmip6.f09_g17.LE2-1001.001, b.e21....   

                                                Run File  
ERA5   [/glade/work/rneale/data/ERA5/VAR_TBD.day.mean...  
MERRA  [/glade/work/rneale/data/MERRA/VAR_TBD.day.mea...  
CESM2  [/glade/campaign/cgd/cesm/CESM2-LE/atm/proc/ts...  
block_met

### Add the MDTF case(s)

In [ ]:
# Add the MDTF case(s) to the block_meta table (data passed as opened datasets)
import pandas as pd

for case_name, c in run.case_list.items():
    new_row = {'Ensemble Type': 'mdtf', 'Ensemble Size': 1,
               'Start Year': standalone_html.year_of(c['startdate']),
               'End Year': standalone_html.year_of(c['enddate']),
               'Run Name': case_name, 'Run File': case_data[case_name]}
    block_meta = pd.concat([block_meta, pd.DataFrame([new_row], index=[case_name])])

print(f' {block_meta=}')


### Daily Z500 for every dataset (opened when first needed)

In [ ]:
# The case(s) are already open (Section 2). An obs or ensemble dataset is only opened if one of its
# blocking frequencies is not in the cache or the POD's obs data ($OBS_DATA), e.g. other years or
# members, or MDTF_BLOCKING_RECOMPUTE. That needs its raw daily Z500, read through MDTF_BLOCKING_RAW_CATALOG.
cam_var = 'Z500'  # name of the variable in the datasets
ensemble_ds = block_utils.EnsembleData(block_meta,cam_var,block_season,block_diag_hem)
for name in block_meta.index:
    if isinstance(block_meta.loc[name]['Run File'], xr.Dataset):
        ensemble_ds[name] = block_meta.loc[name]['Run File']
ens_names = list(block_meta.index)
print(f"Datasets: {ens_names}")

### Blocking frequency: 1D index (season)

In [ ]:
# 1D blocking frequency by longitude, for the season. Cached results are reused
# (see MDTF_BLOCKING_RECOMPUTE and MDTF_BLOCKING_CACHE_DIR);
# shipped obs/ensemble results are read from $OBS_DATA.
importlib.reload(block_utils) # pick up edits to blocking_utils.py
ensemble_block_1d = block_utils.block_z500_freq(block_meta,ensemble_ds,fout_dir,block_season,block_diag='1D',
                                                cache_dir=cache_dir,recompute=recompute,ref_dir=ref_dir)

In [35]:
print(ens_names)


['ERA5', 'MERRA', 'CESM2', 'timeslice_mdtfv3']


In [36]:

for i in range(len(ens_names)):
    ens_name = ens_names[i]
    print(f"++++ Ensemble {i}: {ens_name}")
    print(f"      {ensemble_block_1d[ens_name].dims}")



++++ Ensemble 0: ERA5
      ('name', 'lon')
++++ Ensemble 1: MERRA
      ('name', 'lon')
++++ Ensemble 2: CESM2
      ('name', 'lon')
++++ Ensemble 3: timeslice_mdtfv3
      ('name', 'lon')


### Blocking frequency: 2D index (season)

In [ ]:
# 2D blocking frequency (Davini et al. 2012), for the season
ensemble_block_2d = block_utils.block_z500_freq(block_meta,ensemble_ds,fout_dir,block_season,block_diag='2D',
                                                cache_dir=cache_dir,recompute=recompute,ref_dir=ref_dir)

### Blocking frequency: annual cycle (1D index by day of year)

In [ ]:
# Annual cycle: daily climatology (day of year x longitude) of the 1D blocking flag, all seasons
ensemble_block_ac = block_utils.block_z500_freq(block_meta,ensemble_ds,fout_dir,'ANNCYC',block_diag='1DAC',
                                                cache_dir=cache_dir,recompute=recompute,ref_dir=ref_dir)

## Section 4: Figures

### Seasonal blocking frequency by longitude (1D index)

In [ ]:
block_figs.block_plot_1d(block_meta,ensemble_block_1d,block_season,fig_out=True,dir_fig=fig_dir,
                         fig_name=f"block_1d_freq_{block_season}.png")


### Blocking frequency maps (2D index): ensemble mean

In [ ]:
block_figs.block_plot_2d(block_meta,ensemble_block_2d,block_season,fig_out=True,dir_fig=fig_dir,ens_plot='av',
                         fig_name=f"block_2d_freq_{block_season}.png")

### Annual cycle (Hovmollers): OBS, model ensembles and each case

In [ ]:
ens_types = block_meta['Ensemble Type']
obs_list  = [e for e in block_meta.index if ens_types[e] == 'obs']
mod_list  = [e for e in block_meta.index if ens_types[e] == 'model']
mdtf_list = [e for e in block_meta.index if ens_types[e] == 'mdtf']

if obs_list:
    block_figs.block_plot_anncycle(block_meta, ensemble_block_ac, obs_list,
                                   'Blocking frequency annual cycle: OBS',
                                   dir_fig=os.path.join(pod_work_dir, 'obs'), fig_name='block_freq_anncycle.OBS.png')
if mod_list:
    block_figs.block_plot_anncycle(block_meta, ensemble_block_ac, mod_list,
                                   'Blocking frequency annual cycle: model ensembles',
                                   dir_fig=fig_dir, fig_name='block_freq_anncycle.ENSEMBLES.png')
for case_name in mdtf_list:
    block_figs.block_plot_anncycle(block_meta, ensemble_block_ac, [case_name],
                                   f'Blocking frequency annual cycle: {case_name}',
                                   dir_fig=fig_dir, fig_name=f'block_freq_anncycle.{case_name}.png')


In [40]:
fout_dir

'/glade/work/bundy/mdtf/outdir/20250429/blocking_nb'

## Section 5: Output
Stand-alone runs write the same output as an MDTF framework run (`<WORK_DIR>/MDTF_output/`: `index.html`,
POD page, logs, `case_info.yml`, `config_save.json`; `MDTF_output.tar` if `make_variab_tar`), using the
framework's own html code. Framework runs skip this; the framework builds the output itself.

In [ ]:
make_output = True
if make_output and run.mode != "framework":
    if run.mode == "interactive":
        # Headless and framework runs save the executed notebook as <notebook>_ipynb.html
        # (the page's "Executed notebook" link). Here, write it from the SAVED notebook,
        # before the tar is made. Save the notebook (Ctrl-S) before running this cell.
        standalone_html.export_saved_notebook(os.path.join(pod_code_dir, NOTEBOOK),
                                              pod_work_dir, since=RUN_START)
    index = standalone_html.make_standalone_output(
        run, settings_dict, driver=os.path.join(pod_code_dir, NOTEBOOK), data_files=data_files)